# 01 — Explore raw pcap: calibrate `attack_schedule.yaml`

Phase 1 blocking step (see plan's "Ground-truth labeling strategy"): the
descriptor paper only states attack dates as "26 August" / "29 August",
with no year or timezone. This notebook reads real first/last packet
timestamps from the raw pcapng captures to anchor those dates, and
identifies the shared victim/MEC IP (Level-2 labeling evidence) and the
UE/attacker address ranges empirically, before `configs/attack_schedule.yaml`
is trusted for labeling.

**Findings already baked into `configs/attack_schedule.yaml` from a prior
run of this notebook's logic** (re-running the cells below should reproduce
them):

- Packet timestamps are plain UTC epoch seconds.
- `SYNScan_BS1.pcapng` and `SYNScan_BS2.pcapng` both span
  `2022-08-26 09:30:02–09:38:44 UTC` — confirms `year=2022` for "26 August",
  verified independently on two files/base stations.
- `SSH_BS1.pcapng` (pure benign) spans `2022-09-07 11:39–12:35 UTC` — a
  different day, as expected since it has no attack schedule row.
- Converting to local time (Europe/Helsinki, EEST=UTC+3) gives
  `12:30:02–12:38:44`, which does **not** line up cleanly with Table III's
  published `12:20–12:30` SYN Scan window — it starts ~10 minutes after the
  table's window ends. Conclusion: the published schedule is *planned*
  timing, not an exact-to-the-minute log; Level 1 (schedule) must be treated
  as coarse evidence only.
- Victim IP `10.41.150.68` receives ~10,000 SYN packets in **both** BS1 and
  BS2 SYN Scan captures, vs. single digits to any other address — a very
  clean, cross-BS-consistent Level-2 signal, matching the descriptor paper's
  single-shared-victim topology (Fig. 2).
- Attacker subnet observed: `10.155.15.0/24`; BS1 attacker `10.155.15.1`,
  BS2 attacker `10.155.15.14` (may vary by session; not as reliable as the
  fixed victim IP).

In [ ]:
from collections import Counter
from datetime import UTC, datetime

from agente_5g.parsers.scapy_parser import ScapyPacketParser
from agente_5g.settings import PROJECT_ROOT

RAW_DIR = PROJECT_ROOT / "data" / "raw"
parser = ScapyPacketParser()


## Step 1 — SSH_BS1.pcapng (pure benign session): first/last timestamp

In [ ]:
path = RAW_DIR / "BS1" / "SSH_BS1.pcapng"
first_ts = last_ts = None
n = 0
for rec in parser.parse_file(path, base_station="BS1", source_attack_type="SSH"):
    if first_ts is None:
        first_ts = rec.timestamp
    last_ts = rec.timestamp
    n += 1

print(f"packets: {n}")
print(f"first: {datetime.fromtimestamp(first_ts, tz=UTC)} UTC")
print(f"last:  {datetime.fromtimestamp(last_ts, tz=UTC)} UTC")
print(f"duration_s: {last_ts - first_ts:.1f}")


## Step 2 — SYNScan_BS1.pcapng: timestamp calibration + victim/attacker IP identification

Full-file scan (moderate size, ~76MB, ~30s at ~2400 pkt/s with the Scapy
streaming parser). We look at TCP SYN (no ACK) packets specifically, since a
SYN scan's destination-IP distribution should be extremely skewed toward the
victim being scanned.

In [ ]:
path = RAW_DIR / "BS1" / "SYNScan_BS1.pcapng"
first_ts = last_ts = None
n = is_gtp = 0
syn_dst: Counter = Counter()
syn_src: Counter = Counter()
inner_src_all: Counter = Counter()

for rec in parser.parse_file(path, base_station="BS1", source_attack_type="SYNScan"):
    if first_ts is None:
        first_ts = rec.timestamp
    last_ts = rec.timestamp
    n += 1
    if rec.is_gtp:
        is_gtp += 1
        if rec.inner_src_ip:
            inner_src_all[rec.inner_src_ip] += 1
        if rec.tcp_syn and not rec.tcp_ack:
            if rec.inner_dst_ip:
                syn_dst[rec.inner_dst_ip] += 1
            if rec.inner_src_ip:
                syn_src[rec.inner_src_ip] += 1

print(f"packets: {n}  (is_gtp: {is_gtp})")
print(f"first: {datetime.fromtimestamp(first_ts, tz=UTC)} UTC")
print(f"last:  {datetime.fromtimestamp(last_ts, tz=UTC)} UTC")
print(f"duration_s: {last_ts - first_ts:.1f}")
print()
print("Top SYN destination IPs (candidate victim):", syn_dst.most_common(5))
print("Top SYN source IPs (candidate attacker):", syn_src.most_common(5))
print("Top inner src IPs overall:", inner_src_all.most_common(10))


## Step 3 — SYNScan_BS2.pcapng: cross-check the victim IP is the same across base stations

Bounded to the first 60k packets (sufficient for a dominant-destination
signal without a full ~30s scan).

In [ ]:
path = RAW_DIR / "BS2" / "SYNScan_BS2.pcapng"
syn_dst_bs2: Counter = Counter()
syn_src_bs2: Counter = Counter()

for rec in parser.parse_file(
    path, base_station="BS2", source_attack_type="SYNScan", max_packets=60000
):
    if rec.is_gtp and rec.tcp_syn and not rec.tcp_ack:
        if rec.inner_dst_ip:
            syn_dst_bs2[rec.inner_dst_ip] += 1
        if rec.inner_src_ip:
            syn_src_bs2[rec.inner_src_ip] += 1

print("Top SYN destination IPs (BS2):", syn_dst_bs2.most_common(5))
print("Top SYN source IPs (BS2):", syn_src_bs2.most_common(5))


## Conclusion

If the top SYN-destination IP matches between the BS1 and BS2 runs above
(expected: `10.41.150.68` per the prior run recorded at the top of this
notebook), that confirms the shared-victim topology and
`configs/attack_schedule.yaml`'s `victim_ip` field. If it does not match —
e.g. the dataset copy on disk differs from the one calibrated against —
update `configs/attack_schedule.yaml` (`victim_ip`, `year`, and the
calibration note) accordingly before trusting Phase 4 labeling output, and
flip `calibrated: true` back to `false` until re-verified.

Because the published per-minute schedule windows were found not to align
exactly with observed capture timestamps (~10 minute drift, see the SYN Scan
finding above), Phase 4's `preprocessing/labeling.py` should treat
`attack_window`/`session_window` from `attack_schedule.yaml` as an
approximate, LOW-confidence-only signal ("this file plausibly contains an
attack around this time of day"), and rely primarily on Level 2 (victim IP)
and Level 3 (traffic pattern) to locate the actual attack sub-window within
each file's own observed timestamp span.